# Run Jasna in Google Colab (free tier)

This notebook runs the official Jasna Linux NVIDIA release with the **free BasicVSR++ model**. It does not require Docker, a license code, or a paid GPU service.

> **Free Colab GPUs are not guaranteed.** Choose **Runtime → Change runtime type → T4 GPU** if it is offered. Jasna requires an NVIDIA GPU with compute capability 7.5+ and Linux driver 580+. Colab may assign unsupported hardware, run out of free GPU quota, or have an older host driver; the checks below will stop with an explanation in that case. This notebook cannot upgrade Colab's host driver or bypass Jasna's checks.

Keep Jasna installed under `/content` (the VM's local disk), not Google Drive. This downloads several GB, and the first model/engine setup takes time. Jasna's cloud guide recommends at least 80 GB of local disk; a free Colab runtime may not have enough space, and the notebook cannot increase its disk. The Colab VM is temporary: download/copy your result to Drive before the session ends. For privacy, run this notebook only in your own Colab account and do not share a notebook containing private media paths.

## 1. Check the free GPU and driver

Select a GPU runtime first (`Runtime → Change runtime type → T4 GPU`, if available), then run this cell. If Colab gives you a CPU runtime or a driver below Jasna's minimum, stop here; a free session may work later when a compatible GPU is available.

In [ ]:
import re, subprocess, torch

if not torch.cuda.is_available():
    raise RuntimeError('No NVIDIA GPU is attached. In Colab, choose Runtime → Change runtime type → a GPU, then reconnect.')
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
driver = subprocess.check_output(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], text=True).splitlines()[0].strip()
driver_tuple = tuple(map(int, re.findall(r'\d+', driver)[:3]))
print(f'GPU: {gpu_name} | compute capability: {capability[0]}.{capability[1]} | driver: {driver}')
if capability < (7, 5):
    raise RuntimeError('Jasna requires compute capability 7.5 or newer; this Colab GPU is too old.')
if driver_tuple < (580,):
    raise RuntimeError(f'Jasna requires Linux NVIDIA driver 580+; this Colab runtime has {driver}. Colab controls the host driver, so try a later compatible runtime or another GPU service.')
print('Basic hardware checks passed. Jasna performs its own additional compatibility checks when it starts.')

## 2. Install Jasna locally in this Colab session

This fetches the official `v0.12.3` Linux NVIDIA release parts from GitHub, unpacks them to `/content`, and installs the small system libraries Jasna needs. The archive is multi-gigabyte; keep this cell running until it finishes.

In [ ]:
import json, pathlib, re, shlex, shutil, subprocess, urllib.request

JASNA_TAG = 'v0.12.3'  # Pinned official release; change only to a published Linux NVIDIA release.
root = pathlib.Path('/content')
install_root = root / 'jasna-install'
install_root.mkdir(exist_ok=True)
api_url = f'https://api.github.com/repos/Kruk2/jasna/releases/tags/{JASNA_TAG}'
with urllib.request.urlopen(api_url) as response:
    release = json.load(response)
version = release['tag_name'].removeprefix('v')
part_pattern = re.compile(r'^jasna-linux-nvidia-' + re.escape(version) + r'\.tar\.zst\.part\d+$')
parts = sorted((item for item in release.get('assets', []) if part_pattern.fullmatch(item['name'])), key=lambda item: item['name'])
if not parts:
    raise RuntimeError(f'No official Linux NVIDIA archive parts found for {JASNA_TAG}.')

subprocess.run(['apt-get', 'update', '-qq'], check=True)
for package in ('zstd', 'libgl1', 'libglib2.0-0'):
    subprocess.run(['apt-get', 'install', '-y', '-qq', package], check=True)

archive_dir = root / 'jasna-release-parts'
archive_dir.mkdir(exist_ok=True)
for item in parts:
    destination = archive_dir / item['name']
    if not destination.exists() or destination.stat().st_size != item['size']:
        print(f'Downloading {item["name"]} ({item["size"] / 1e9:.1f} GB) ...')
        subprocess.run(['wget', '--show-progress', '--progress=dot:giga', '-O', str(destination), item['browser_download_url']], check=True)

extraction_dir = install_root / 'unpacked'
extraction_dir.mkdir(exist_ok=True)
binary_dir = install_root / f'jasna-linux-nvidia-{version}'
if not (binary_dir / 'jasna').is_file():
    quoted_parts = ' '.join(shlex.quote(str(archive_dir / item['name'])) for item in parts)
    command = f'cat {quoted_parts} | zstd -dc | tar -xpf - --same-owner -C {shlex.quote(str(extraction_dir))}'
    subprocess.run(['bash', '-o', 'pipefail', '-c', command], check=True)
    extracted = extraction_dir / f'jasna-linux-nvidia-{version}'
    if not (extracted / 'jasna').is_file():
        raise RuntimeError(f'Expected Jasna executable not found in {extracted}.')
    if binary_dir.exists():
        shutil.rmtree(binary_dir)
    extracted.rename(binary_dir)
jasna = binary_dir / 'jasna'
print(f'Jasna {JASNA_TAG} is ready at {jasna}')

## 3. Connect Drive and choose a video

Upload your source video to your own Google Drive, then set `drive_video` below to its path. The notebook copies it onto Colab's local disk before processing (the VM may have limited temporary storage).

In [ ]:
from google.colab import drive
import pathlib, shutil
drive.mount('/content/drive')

drive_video = pathlib.Path('/content/drive/MyDrive/jasna/test.mp4')  # Video supplied by the user.
if not drive_video.is_file():
    raise FileNotFoundError(f'Update drive_video to an existing file in your Drive: {drive_video}')
local_input = pathlib.Path('/content/input_video' + drive_video.suffix.lower())
shutil.copy2(drive_video, local_input)
local_output = pathlib.Path('/content/restored_video.mkv')
drive_output = pathlib.Path('/content/drive/MyDrive/restored_video.mkv')
print(f'Input copied to {local_input}; output will be copied to {drive_output}')

## 4. Restore with the free model

This runs free BasicVSR++ without a supporter license. Colab's GPU allotment can end or disconnect during long jobs; keep a backup of the original video.

In [ ]:
import os

env = os.environ.copy()
env['XDG_CONFIG_HOME'] = '/content/jasna-config'  # Temporary local config; no license code is needed for BasicVSR++.
subprocess.run([
    str(jasna),
    '--input', str(local_input),
    '--output', str(local_output),
    '--restoration-model-name', 'basicvsrpp',
], cwd=str(binary_dir), env=env, check=True)
if not local_output.is_file():
    raise RuntimeError('Jasna finished without producing the expected output file.')
shutil.copy2(local_output, drive_output)
print(f'Restored video saved to Drive: {drive_output}')

## Limitations

- This uses the free BasicVSR++ restoration only. It does not enable paid/supporter models.
- Free Colab GPU access, GPU type, driver version, disk space, and session length are controlled by Google and may change. There is no way for this project to guarantee a free GPU or make an incompatible runtime work.
- Jasna's model preparation can take a long time, and a disconnected Colab session loses files under `/content`; copy results to Drive before the session expires.
- Use Jasna's official documentation for its supported GPU/driver list and the latest release details: https://github.com/Kruk2/jasna/blob/main/docs/en/cloud.md